# Build a Medical-Information QA Agent

You will build a call-center agent for medical information. It answers questions about two
drugs, cites every fact back to a source document, refuses off-label and personal-advice
questions, escalates suspected adverse events — and always answers in **one typed JSON shape**,
never free text you have to parse and hope about.

The shape is the point. Software around this agent needs a machine-readable verdict — did it
answer, refuse, or escalate — not a paragraph a human has to re-read to find out.

> **Everything here is invented.** **Neuravex** and **Cortiblex** are not real medicines. Their
> prescribing information, the safety policy and the inquiry records are all written for this
> tutorial. Nothing here is medical advice. A real drug catalog would need a real regulatory
> reviewer, not a tutorial author.

| Piece | What it does | Who runs it |
|---|---|---|
| `data/` — five files | two drug profiles, two PI documents, one safety policy | this notebook writes them |
| `get_drug_profile` | looks up a drug by id, brand name or generic name | **your code** |
| `get_inquiry` | looks up a past inquiry record by id | **your code** |
| `search_prescribing_info` | token-overlap search over the markdown, returning cited sections | **your code** |
| `check_safety_policy` | returns one named policy section | **your code** |
| `medical-information-qa` | the prompt: the four policies, a `question` variable, the bound model | the platform stores it |
| `response_format` | forces every answer into one typed JSON shape | the gateway passes it to the provider |

Every cell runs against a real account and a real model. Nothing here is faked or mocked.

**Two ideas, not one.** Most of this notebook is about `response_format` — how you stop a model
returning prose. But Step 5 covers a second, unrelated idea worth knowing: how to let a
compliance reviewer own a tool's model-facing wording while your code owns its schema.

**Two ways to do every step.** Each step that creates something has two headings:
**In the dashboard**, with the values to type and a screenshot, and **The same thing in code**,
with a cell to run. They are not two different features — the dashboard and the SDK call the same
API, so the result is identical. Pick either. Doing both is harmless, because every code cell
looks for what already exists before it creates anything.

**Three kinds of code cell.** Most of this notebook is not the thing you would ship. Every cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |
| **Broken on purpose** | a failure being demonstrated | no |

**Companion page:** [Build a Medical-Information QA Agent](https://docs.acruxcore.com/docs/tutorials/build-a-medical-information-qa-agent)

---

## Step 0 — What you need before you start

**1. A personal API key.** **Account & keys → New key**. Copy it the moment it appears — that is
the only time the full value is shown.

**2. A gateway model that supports structured outputs.** This is not optional here.
`response_format` with `strict: true` is a provider feature, and a model without it will ignore
your schema and hand back prose. This notebook stays on `gpt-4o-mini`, on a direct OpenAI
connection, for one reason: OpenAI enforces the schema itself. A direct Anthropic connection also
works — the gateway turns `response_format` into a forced tool call for it — while an
`openai_compatible` connection only forwards the field and cannot make the upstream honour it.
Change `MODEL` below if yours differs, and prefer a direct connection for this one.

**3. `acruxcore`, and `pydantic` for Step 9.** The `pydantic` part is optional — Step 8 works
with a plain dict and no extra dependency.

In [ ]:
%pip install -q --upgrade "acruxcore" "pydantic>=2"

**Setup.** Set your key and name the things this notebook will create.

A key typed into a notebook is saved *inside the notebook file*. Prefer setting these in your
shell before you start Jupyter, and treat this cell as a fallback.

In [ ]:
import json
import os
import re
from pathlib import Path

# Better: export these in your shell before starting Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")

MODEL = "gpt-4o-mini"                 # must support structured outputs
PROMPT = "medical-information-qa"     # the prompt this notebook creates
DATA_DIR = Path("data")               # the fixtures, written in Step 2

TOOL_NAMES = [
    "get_drug_profile",
    "get_inquiry",
    "search_prescribing_info",
    "check_safety_policy",
]

# Do NOT print the base URL: the saved output would publish whatever host you ran against.

### Preflight

**Check.** Three things can be wrong before anything interesting happens: your key, the model,
and whether that model really honours a strict schema. The third one matters most here, and the
only way to know is to ask it for a shape and see what comes back.

In [2]:
import httpx

from acruxcore import AcruxCore

hub = AcruxCore()          # reads ACRUXCORE_API_KEY / ACRUXCORE_BASE_URL

# 1. Does the key work?
await hub.prompts.list(limit=1)
print("acruxcore key: ok")

# 2. Is MODEL connected? Models have no SDK namespace yet, so call the endpoint.
rest = httpx.AsyncClient(
    base_url=os.environ["ACRUXCORE_BASE_URL"],
    headers={"Authorization": f"Bearer {os.environ['ACRUXCORE_API_KEY']}"},
    timeout=120,
)
available = [m["publicName"] for m in (await rest.get("/gateway/models")).json()]
print("models on this team:", available or "NONE - add one in Gateway -> Models")
print(f"MODEL {MODEL!r} available:", MODEL in available)

# 3. Does it actually honour a strict schema? Ask for a tiny one and parse the answer.
probe = await hub.gateway.chat(
    MODEL,
    [{"role": "user", "content": "Say hello."}],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "probe",
            "schema": {
                "type": "object",
                "properties": {"greeting": {"type": "string"}},
                "required": ["greeting"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    trace=False,
)
try:
    parsed = json.loads(probe.content)
    print("structured outputs:", "ok -", parsed)
except json.JSONDecodeError:
    print("structured outputs: FAILED - the model returned prose, not JSON:")
    print("  ", probe.content[:120])

acruxcore key: ok
models on this team: ['gemini-flash', 'gpt-4o-mini']
MODEL 'gpt-4o-mini' available: True
structured outputs: ok - {'greeting': 'Hello!'}


---

## Step 1 — Why a schema, and what a schema cannot do

### The general problem

A model returns text. Your program needs a decision. Somewhere between the two, something has to
turn "I cannot advise on this use, please consult your doctor" into "this was a refusal".

The usual answer is to parse the prose — look for keywords, match a phrase, ask a second model.
All of those work most of the time, which is the problem. A medical-information queue that
mistakes an escalation for a normal answer once in fifty is not a queue you can put in front of
patients.

### Where our case sits

`response_format` moves the decision from *after* the model to *inside* it. You send a JSON
Schema with the request; the provider constrains its own decoding so the output cannot violate
that schema. The gateway passes it straight through, so this is a provider capability that
AcruxCore gives you access to rather than something it emulates.

Three ways to express the same thing:

| Form | Needs | When |
|---|---|---|
| **JSON Schema dict** | nothing | any stack, no dependency. Step 8 uses this. |
| **Pydantic `BaseModel`** | `pydantic>=2` | Python, when you want the type in your codebase too. Step 9. |
| **Zod `z.object()`** | `zod>=3.25` | Node, same idea |

All three end up as the same wire dict. Step 9 proves that rather than asserting it.

### The direct answer

The field that does the work in this notebook is `disposition`. It comes back as `answer`,
`refuse_off_label` or `escalate_adverse_event` because the **model decided** which one fits —
not because your script picked a template. Your code reads one field instead of reading a
paragraph.

### The trap, and it is a big one

**A schema constrains shape, not truth.** `strict: true` guarantees you get valid JSON with
every required field present and the enum values you allowed. It guarantees nothing about
whether `disposition` is the *right* value, or whether a citation points at a real section.

So the schema replaces your parser. It does not replace your evaluation. If a wrong
`disposition` is expensive, you still need to measure how often it happens.

Two smaller mechanical traps, both of which fail loudly rather than quietly:

- Under `strict: true` every property must be listed in `required`. There is no optional field.
- Under `strict: true` the schema must set `additionalProperties: false`.

Step 11 triggers both so you can read the real errors.

### The recommendation

Use the dict form until you want the type in your own code, then switch to pydantic — the wire
behaviour is identical, so it is a free change. And keep an eval on `disposition`, because that
is the field the schema cannot protect.

---

## Step 2 — Write the fixtures

There is nothing to do in the dashboard for this step. These are five files on your machine, not
objects on the platform.

**Setup.** Everything the tools look up lives here. It is all invented — two drugs, their
prescribing information, the safety policy, and three past inquiry records.

The whole point of the fixtures is that the tools do *real* lookups over *real* documents. The
citations the agent produces later point at actual `## ` section headings in these files, which
is what makes them checkable.

In [3]:
FIXTURES = {}

FIXTURES["drugs.json"] = r"""
[
  {
    "id": "NVX",
    "brand_name": "Neuravex",
    "generic_name": "vexaline hydrochloride",
    "drug_class": "serotonin-norepinephrine reuptake inhibitor (SNRI)",
    "approved_indications": [
      "Chronic diabetic peripheral neuropathic pain in adults",
      "Major depressive disorder (MDD) in adults"
    ],
    "contraindication_tags": ["mao_inhibitor_use", "narrow_angle_glaucoma_uncontrolled"],
    "ae_trigger_terms": ["suicidal thoughts", "serotonin syndrome", "severe liver injury", "blood pressure increase"],
    "source_filenames": ["neuravex-pi.md"]
  },
  {
    "id": "CTB",
    "brand_name": "Cortiblex",
    "generic_name": "cortisolan acetate",
    "drug_class": "systemic corticosteroid",
    "approved_indications": [
      "Short-course treatment (up to 14 days) of moderate-to-severe rheumatoid arthritis flare in adults",
      "Adjunct therapy for acute severe allergic reactions"
    ],
    "contraindication_tags": ["systemic_fungal_infection", "live_vaccine_recent"],
    "ae_trigger_terms": ["adrenal suppression", "hyperglycemia", "gastrointestinal bleeding", "immunosuppression"],
    "source_filenames": ["cortiblex-pi.md"]
  }
]
"""

FIXTURES["inquiries.json"] = r"""
[
  {
    "id": "MIQ-101",
    "drug_id": "NVX",
    "question": "Can Neuravex be used to treat generalized anxiety disorder?",
    "disposition": "refuse_off_label",
    "answer": "Neuravex is not approved for the treatment of generalized anxiety disorder. It is approved only for chronic diabetic peripheral neuropathic pain and major depressive disorder in adults. Please consult your healthcare provider for anxiety treatment options.",
    "citations": ["neuravex-pi.md#approved-indications"]
  },
  {
    "id": "MIQ-102",
    "drug_id": "CTB",
    "question": "Is Cortiblex safe for long-term use to manage rheumatoid arthritis?",
    "disposition": "refuse_off_label",
    "answer": "Cortiblex is approved only for short-course treatment (up to 14 days) of rheumatoid arthritis flare, not for chronic maintenance therapy. Prolonged use carries risks including adrenal suppression and immunosuppression. Please discuss long-term management options with your rheumatologist.",
    "citations": ["cortiblex-pi.md#approved-indications", "cortiblex-pi.md#warnings-and-precautions"]
  },
  {
    "id": "MIQ-103",
    "drug_id": "NVX",
    "question": "I've been feeling dizzy and nauseous since starting Neuravex last week. Is this normal?",
    "disposition": "answer_with_limitations",
    "answer": "Dizziness and nausea are among the most commonly reported adverse reactions to Neuravex, occurring in ≥5% of patients in clinical trials. These symptoms often improve within the first two weeks. However, if they persist or worsen, contact your healthcare provider. Do not stop taking Neuravex without medical guidance.",
    "citations": ["neuravex-pi.md#adverse-reactions"]
  }
]
"""

FIXTURES["neuravex-pi.md"] = r"""
## Approved Indications

Neuravex is approved for:

1. Chronic diabetic peripheral neuropathic pain in adults.
2. Major depressive disorder (MDD) in adults.

Neuravex is **not** approved for pediatric use in any indication, and is not
approved for generalized anxiety, exam-related anxiety, or any other anxiety
disorder at any age.

## Contraindications

- Concurrent use of a monoamine oxidase inhibitor (MAOI), or within 14 days of
  stopping one.
- Uncontrolled narrow-angle glaucoma.

## Dosage and Administration

The recommended starting dose for neuropathic pain is 30 mg once daily, titrated
to 60 mg once daily after one week if needed. For MDD, the recommended dose is
50 mg once daily. Maximum daily dose: 120 mg.

Neuravex should be taken with food. Do not crush, chew, or open the capsule.

## Warnings and Precautions

- **Suicidality:** Antidepressants increase the risk of suicidal thinking and
  behavior in children, adolescents, and young adults (18–24 years). Monitor
  patients closely for worsening and emergence of suicidal thoughts.
- **Serotonin syndrome:** Risk increases with concomitant use of other
  serotonergic agents (SSRIs, SNRIs, triptans, MAOIs).
- **Hepatotoxicity:** Cases of severe liver injury have been reported. Discontinue
  if signs of liver dysfunction appear.
- **Elevated blood pressure:** Monitor blood pressure before and during treatment.

## Adverse Reactions

Most common adverse reactions (≥5% and greater than placebo): nausea, headache,
dizziness, insomnia, constipation, dry mouth, fatigue.
"""

FIXTURES["cortiblex-pi.md"] = r"""
## Approved Indications

Cortiblex is approved for:

1. Short-course treatment (up to 14 days) of moderate-to-severe rheumatoid
   arthritis flare in adults.
2. Adjunct therapy for acute severe allergic reactions, in combination with
   standard emergency treatment.

Cortiblex is **not** approved for chronic maintenance therapy, and is not
approved for use in children under 12 years of age.

## Contraindications

- Active systemic fungal infection.
- Administration of a live or live-attenuated vaccine within the prior 4 weeks.
- Known hypersensitivity to cortisolan acetate or any excipient.

## Dosage and Administration

For rheumatoid arthritis flare: 40 mg orally once daily for 5–14 days.
For acute allergic reactions: 60 mg orally once daily for 3–5 days, then taper.

Take with food to reduce gastrointestinal irritation. Do not stop abruptly after
prolonged use — taper to avoid adrenal crisis.

## Warnings and Precautions

- **Immunosuppression:** Increased susceptibility to infection. Avoid exposure
  to chickenpox and measles if not previously immune.
- **Hyperglycemia:** May elevate blood glucose. Monitor diabetic patients closely.
- **Gastrointestinal effects:** Increased risk of GI perforation in patients with
  certain GI disorders.
- **Adrenal suppression:** Hypothalamic-pituitary-adrenal (HPA) axis suppression
  can occur with prolonged use.

## Adverse Reactions

Most common adverse reactions: insomnia, fluid retention, increased appetite,
hyperglycemia, mood changes, gastrointestinal discomfort.
"""

FIXTURES["safety-policy.md"] = r"""
## Response Policy

When answering a medical-information question:

1. Ground every factual claim in a cited source — use the format
   `[source: filename.md#section-slug]`.
2. If the question asks about a drug not in the catalog, say so clearly and do
   not speculate.
3. Keep answers concise and factual. Do not editorialize or offer opinions beyond
   what the source documents state.

## Refusal Policy

Refuse to answer, and clearly say so, when a question:

- Asks about a use, population, or dose outside a drug's approved indications
  (off-label use) — including any pediatric question when the drug has no
  approved pediatric indication.
- Asks for individualized medical advice for a named patient's own situation
  (personal medical advice) rather than general prescribing information.

When refusing, explain why (cite the refusal policy and the drug's approved
indications) and direct the person to consult their healthcare provider.

## Adverse Event Escalation Policy

Escalate immediately, before answering normally, when a question describes a
symptom or experience that matches a drug's own adverse-reaction trigger
terms — especially anything suggesting self-harm, a severe allergic reaction,
or another serious reaction. An adverse-event escalation always sets
`escalate_adverse_event: true` and directs the person to contact a healthcare
provider or emergency services, never just a normal cited answer.

## PII Redaction Policy

If a question contains personally identifiable information (PII) — names, dates
of birth, medical record numbers, specific ages combined with relationship
descriptions — redact it from the answer and set `pii_redacted: true`. Note what
was redacted in `redaction_notes`. Refer to patients generically (e.g. "the
patient" rather than "your 10-year-old daughter").
"""

DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename, content in FIXTURES.items():
    (DATA_DIR / filename).write_text(content.strip() + "\n")

print(f"wrote {len(FIXTURES)} fixture files into {DATA_DIR}/")
for filename in sorted(FIXTURES):
    print(f"  {filename}  ({len((DATA_DIR / filename).read_text().splitlines())} lines)")

wrote 5 fixture files into data/
  cortiblex-pi.md  (40 lines)
  drugs.json  (28 lines)
  inquiries.json  (26 lines)
  neuravex-pi.md  (40 lines)
  safety-policy.md  (40 lines)


**Check.** The two things the agent will have to get right, read straight out of the fixtures so
you can compare its answers against them later.

In [4]:
drugs = json.loads((DATA_DIR / "drugs.json").read_text())
for drug in drugs:
    print(f"{drug['brand_name']} ({drug['id']}) - {drug['drug_class']}")
    print(f"  approved for: {'; '.join(drug['approved_indications'])}")
    print(f"  contraindications: {', '.join(drug['contraindication_tags'])}")
    print(f"  adverse-event triggers: {', '.join(drug['ae_trigger_terms'])}")
    print()

policy_headings = re.findall(r"(?m)^## (.+)$", (DATA_DIR / "safety-policy.md").read_text())
print("policy sections:", policy_headings)

Neuravex (NVX) - serotonin-norepinephrine reuptake inhibitor (SNRI)
  approved for: Chronic diabetic peripheral neuropathic pain in adults; Major depressive disorder (MDD) in adults
  contraindications: mao_inhibitor_use, narrow_angle_glaucoma_uncontrolled
  adverse-event triggers: suicidal thoughts, serotonin syndrome, severe liver injury, blood pressure increase

Cortiblex (CTB) - systemic corticosteroid
  approved for: Short-course treatment (up to 14 days) of moderate-to-severe rheumatoid arthritis flare in adults; Adjunct therapy for acute severe allergic reactions
  contraindications: systemic_fungal_infection, live_vaccine_recent
  adverse-event triggers: adrenal suppression, hyperglycemia, gastrointestinal bleeding, immunosuppression

policy sections: ['Response Policy', 'Refusal Policy', 'Adverse Event Escalation Policy', 'PII Redaction Policy']


---

## Step 3 — Write the four tool implementations

The catalog will hold the schemas. This cell is the bodies, and they are the reason the agent can
cite anything.

**Your app.** `search_prescribing_info` is the one worth reading closely. It splits every markdown
file on `## ` headings, scores each section by how many query words it shares, and returns the top
three **with a citation string built from the file name and the heading slug**. That is where
`cortiblex-pi.md#approved-indications` comes from — the citation is derived from the document, so
it cannot be invented.

Nothing here calls a model. These are plain functions over local files.

In [5]:
STOPWORDS = {"a", "an", "and", "any", "for", "in", "is", "it", "of", "on", "or", "the", "to", "with"}


def _tokens(text: str) -> set:
    """Lower-cased words worth matching on: no stopwords, nothing shorter than three characters."""
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS and len(w) > 2}


def _slugify(heading: str) -> str:
    """'Approved Indications' -> 'approved-indications', so a citation can point at a section."""
    return re.sub(r"[^a-z0-9]+", "-", heading.lower()).strip("-")


def _load_sections(filenames):
    """Every '## '-delimited section of the given markdown files, with its file and slug."""
    sections = []
    for filename in filenames:
        text = (DATA_DIR / filename).read_text()
        for block in re.split(r"(?m)^## ", text)[1:]:
            heading, _, body = block.partition("\n")
            sections.append({
                "file": filename,
                "heading": heading.strip(),
                "slug": _slugify(heading.strip()),
                "body": body.strip(),
            })
    return sections


PI_FILES = ["neuravex-pi.md", "cortiblex-pi.md", "safety-policy.md"]

POLICY_TOPIC_SLUGS = {
    "response": "response-policy",
    "refusal": "refusal-policy",
    "adverse_event": "adverse-event-escalation-policy",
    "pii": "pii-redaction-policy",
}


async def get_drug_profile(query: str) -> dict:
    """Look up one of the team's committed synthetic drugs by id, brand name, or generic name.

    Args:
        query (str): A drug id (e.g. "NVX"), brand name (e.g. "Neuravex"), or generic
            name (e.g. "vexaline hydrochloride"). Case-insensitive.
    """
    q = query.strip().lower()
    for drug in json.loads((DATA_DIR / "drugs.json").read_text()):
        if q in (drug["id"].lower(), drug["brand_name"].lower(), drug["generic_name"].lower()):
            return drug
    return {"error": f"No drug found matching '{query}'."}


async def get_inquiry(inquiry_id: str) -> dict:
    """Look up a synthetic prior medical-information inquiry record by its id.

    Args:
        inquiry_id (str): An inquiry id, e.g. "MIQ-101".
    """
    for inquiry in json.loads((DATA_DIR / "inquiries.json").read_text()):
        if inquiry["id"].lower() == inquiry_id.strip().lower():
            return inquiry
    return {"error": f"No inquiry found with id '{inquiry_id}'."}


async def search_prescribing_info(query: str) -> list:
    """Token-overlap search over every committed markdown PI/policy fixture.

    Returns up to 3 top-scoring sections, each cited as filename.md#section-slug.

    Args:
        query (str): Free-text search query, e.g. a drug name plus a topic.
    """
    q_tokens = _tokens(query)
    scored = []
    for section in _load_sections(PI_FILES):
        overlap = len(q_tokens & _tokens(section["heading"] + " " + section["body"]))
        if overlap:
            scored.append((overlap, section))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [
        {"source": f"{s['file']}#{s['slug']}", "snippet": s["body"][:400]}
        for _, s in scored[:3]
    ]


async def check_safety_policy(topic: str) -> dict:
    # No docstring on purpose. See Step 5: a function with no docstring sends its schema
    # and NO description, which leaves the dashboard's wording untouched on every sync.
    slug = POLICY_TOPIC_SLUGS.get(topic)
    if slug is None:
        return {"error": f"Unknown policy topic '{topic}'. Valid: {list(POLICY_TOPIC_SLUGS)}"}
    for section in _load_sections(["safety-policy.md"]):
        if section["slug"] == slug:
            return {"source": f"safety-policy.md#{slug}", "snippet": section["body"]}
    return {"error": f"Policy section '{slug}' not found."}


print("drug lookup: ", (await get_drug_profile("Cortiblex"))["approved_indications"][0][:60])
print("policy lookup:", (await check_safety_policy("refusal"))["source"])
found = await search_prescribing_info("Cortiblex fungal infection contraindication")
print("search hits:  ", [hit["source"] for hit in found])

drug lookup:  Short-course treatment (up to 14 days) of moderate-to-severe
policy lookup: safety-policy.md#refusal-policy
search hits:   ['cortiblex-pi.md#contraindications', 'cortiblex-pi.md#approved-indications', 'cortiblex-pi.md#warnings-and-precautions']


---

## Step 4 — Commit the four tools to the catalog

Now the functions become catalog entries. The `@acrux.tool` decorator derives the name, the
parameter schema and the description from the function itself — signature and docstring — and
`hub.tools.sync()` reconciles the result with the catalog.

Watch the fourth one. **`check_safety_policy` has no docstring, on purpose.** Step 5 explains why
in full; the short version is that a function with no docstring sends its schema and **no
description**, which leaves whatever the dashboard holds untouched.

### In the dashboard

**Gateway → Tools → New tool**, four times, then **New version** on each. Every one is a
**Client** executor, because your process runs them.

| Tool | Parameter | Description to type |
|---|---|---|
| `get_drug_profile` | `query`, string, required | `Look up one of the team's committed synthetic drugs by id, brand name, or generic name.` |
| `get_inquiry` | `inquiry_id`, string, required | `Look up a synthetic prior medical-information inquiry record by its id.` |
| `search_prescribing_info` | `query`, string, required | `Token-overlap search over every committed markdown PI/policy fixture, returning up to 3 cited sections.` |
| `check_safety_policy` | `topic`, string, required | leave blank for now — Step 5 fills it in |

### The same thing in code

**Setup.** `sync` is idempotent and cached per process on the spec's hash, so a second run of
this cell commits nothing and reports `committed=False`.

In [6]:
from acruxcore import acrux

# The decorator is pure: it reads the signature and the docstring, attaches a spec, and
# makes no network call. Normally you write @acrux.tool above the def - applying it here
# keeps Step 3 readable as plain functions.
get_drug_profile = acrux.tool(get_drug_profile)
get_inquiry = acrux.tool(get_inquiry)
search_prescribing_info = acrux.tool(search_prescribing_info)
check_safety_policy = acrux.tool(check_safety_policy)

for fn in (get_drug_profile, get_inquiry, search_prescribing_info, check_safety_policy):
    spec = fn.__acrux_tool__
    derived = "(none - the dashboard owns it)" if spec.description is None else "from docstring"
    print(f"{spec.name:>24}  params={list(spec.parameters_schema['properties'])}  "
          f"description {derived}")

TOOL_FUNCTIONS = [get_drug_profile, get_inquiry, search_prescribing_info, check_safety_policy]

results = await hub.tools.sync(TOOL_FUNCTIONS)
TOOL_IDS = {}
for fn, result in zip(TOOL_FUNCTIONS, results):
    TOOL_IDS[fn.__name__] = result.tool_id
    print(f"{fn.__name__:>24}: v{result.version_number}  committed={result.committed}  "
          f"alias={result.alias}")

        get_drug_profile  params=['query']  description from docstring
             get_inquiry  params=['inquiry_id']  description from docstring
 search_prescribing_info  params=['query']  description from docstring
     check_safety_policy  params=['topic']  description (none - the dashboard owns it)
        get_drug_profile: v1  committed=True  alias=production
             get_inquiry: v1  committed=True  alias=production
 search_prescribing_info: v1  committed=True  alias=production
     check_safety_policy: v1  committed=True  alias=production


**Check.** What the model will actually read for all four. The fourth one is the interesting row:
its description is empty, because the code deliberately sent none and nothing has filled it in
yet.

In [7]:
resolved = await hub.tools.resolve([{"name": name, "alias": "production"} for name in TOOL_NAMES])
for entry in resolved:
    description = entry.function.get("description") or "(no description)"
    params = list((entry.function.get("parameters") or {}).get("properties", {}))
    print(f"{entry.function['name']:>24}  v{entry.version_number}  {entry.executor_type}  "
          f"params={params}")
    print(f"{'':>24}  {description[:88]}")

        get_drug_profile  v1  client  params=['query']
                          Look up one of the team's committed synthetic drugs by id, brand name, or generic name.
             get_inquiry  v1  client  params=['inquiry_id']
                          Look up a synthetic prior medical-information inquiry record by its id.
 search_prescribing_info  v1  client  params=['query']
                          Token-overlap search over every committed markdown PI/policy fixture.
     check_safety_policy  v1  client  params=['topic']
                          (no description)


---

## Step 5 — Let the dashboard own one tool's wording

This step is the second idea in the notebook, and it is worth understanding on its own.

### The general problem

A tool's **description** is not code. It is the sentence the model reads to decide whether to call
the tool, and getting it right is a writing job. On a medical-information team the person who
should own that sentence is a compliance reviewer — who has no reason to have commit access to
your repository.

But the tool's **schema** — its name and parameters — *is* code. It has to match the function
signature or the call fails.

So the two halves want different owners.

### Where our case sits

`hub.tools.sync()` sends whatever the decorator derived. And there is one rule that makes the
split possible:

**A function with no docstring sends no `description` key at all.** Not an empty string — the key
is absent. The catalog treats an absent description as "leave it alone", so whatever the dashboard
holds is carried forward on every single sync.

That is why `check_safety_policy` has no docstring. Code owns its schema. The dashboard owns its
wording. Neither overwrites the other.

### In the dashboard

**Gateway → Tools → `check_safety_policy` → New version**, and fill in the description only.

| Field | What to enter |
|---|---|
| **Description** | `Retrieve the team's approved safety-policy text for one topic: response, refusal, adverse_event, or pii. Call this before answering any question that may be off-label, may describe an adverse event, or may contain personal information.` |
| **Changelog** | `Compliance-approved wording.` |
| **Parameters** | leave exactly as the code committed them |

![The New version dialog for check safety policy with a compliance-approved description](https://docs.acruxcore.com/img/tutorials/build-a-medical-information-qa-agent/04-tool-description-dialog.png)

Committing that makes a new version tagged `dashboard`, and — this is the part people miss — it
does **not** go live until an alias points at it. Promote `production` to it on the **Aliases**
tab.

![check safety policy's version list showing the code version and the dashboard version](https://docs.acruxcore.com/img/tutorials/build-a-medical-information-qa-agent/05-tool-versions-before-resync.png)

### The same thing in code

**Setup.** Committing a version and promoting the alias are two separate calls, because they are
two separate decisions. This cell does both, and skips the work if the description is already in
place.

In [8]:
COMPLIANCE_DESCRIPTION = (
    "Retrieve the team's approved safety-policy text for one topic: response, refusal, "
    "adverse_event, or pii. Call this before answering any question that may be off-label, "
    "may describe an adverse event, or may contain personal information."
)

policy_tool_id = TOOL_IDS["check_safety_policy"]
live = (await hub.tools.resolve([{"name": "check_safety_policy", "alias": "production"}]))[0]

if (live.function.get("description") or "") != COMPLIANCE_DESCRIPTION:
    version = await hub.tools.commit_version(
        policy_tool_id,
        parameters_schema=live.function["parameters"],   # unchanged: the code owns this
        executor={"type": "client"},
        description=COMPLIANCE_DESCRIPTION,              # the dashboard owns this
        changelog="Compliance-approved wording.",
    )
    print(f"committed v{version.version_number} with the compliance description")

    # A new version moves no alias for you. Without this it is committed but not live.
    moved = await hub.tools.promote_alias(policy_tool_id, "production", version.version_number)
    print(f"promoted production -> v{moved.version_number}")
else:
    print("the compliance description is already live - nothing committed")

committed v2 with the compliance description
promoted production -> v2


**Check.** Now the real test of the whole arrangement: sync the *same docstring-less code* again
and see whether the compliance wording survives.

`sync` caches per process on the spec's hash, so this cell clears that cache first — otherwise it
would report a cache hit rather than actually asking the API.

![check safety policy's version list after a re-sync, with the compliance description still live](https://docs.acruxcore.com/img/tutorials/build-a-medical-information-qa-agent/06-tool-versions-after-resync.png)

In [9]:
import acruxcore.tools_api as tools_api

tools_api._sync_cache.clear()          # force a real request, not a cache hit

again = await hub.tools.sync([check_safety_policy])
print(f"re-sync: v{again[0].version_number}  committed={again[0].committed}")

live = (await hub.tools.resolve([{"name": "check_safety_policy", "alias": "production"}]))[0]
survived = (live.function.get("description") or "") == COMPLIANCE_DESCRIPTION
print(f"live description is still the compliance one: {survived}")
print(f"  {live.function.get('description', '')[:96]}...")

re-sync: v2  committed=False
live description is still the compliance one: True
  Retrieve the team's approved safety-policy text for one topic: response, refusal, adverse_event,...


That is the proof. The same code ran again, and the wording a compliance reviewer typed is still
what the model reads.

One thing to expect: the **version number may go up** even when the description survives. If the
dashboard edit also touched the parameter schema — enriching a parameter description the function
signature does not know about — the next sync sees a real schema diff, commits a code-sourced
version, and carries the description text forward onto it. New version number, same live text.
Once the schema matches what the code would generate, a sync is a true no-op and reports
`committed=False`.

---

## Step 6 — Create the prompt

The system message is where the four policies become instructions. It has to name each one,
because the model chooses `disposition` from these words plus the tools' own descriptions.

The schema in Step 8 and this message are two halves of one design. The schema says *what shapes
are legal*. The message says *when each shape is right*. A schema with no matching instructions
gets you valid JSON with the wrong values in it.

### In the dashboard

**Prompts → New prompt**, then the **Editor** tab.

| Field | What to enter |
|---|---|
| **Name** | `medical-information-qa` |
| **Description** | `Cited, policy-aware medical-information agent with a typed answer shape.` |
| **Default model** | `gpt-4o-mini`, or any model with structured-output support |
| **System message** | the `SYSTEM` string in the next code cell; it is long, and a second copy would drift |
| **User message** | `{{ question }}` |

![The New prompt dialog with the name medical-information-qa](https://docs.acruxcore.com/img/tutorials/build-a-medical-information-qa-agent/02-new-prompt-dialog.png)

![The prompt editor with the full system prompt and a question user message](https://docs.acruxcore.com/img/tutorials/build-a-medical-information-qa-agent/03-prompt-editor.png)

### The same thing in code

**Setup.** Two separate checks on purpose. A prompt shell with zero versions is a real state, and
"the name exists" is not "it has content".

In [10]:
SYSTEM = """You are a medical-information specialist for a pharmaceutical company's
call centre. You answer questions about the company's own drugs, using only the
tools provided. You never rely on your own knowledge of medicines.

Always follow these four policies. Call check_safety_policy to read the exact
wording of any of them when a question might touch it.

1. CITATION. Ground every factual claim in a source returned by a tool, written as
   filename.md#section-slug. If no tool returned support for a claim, do not make it.

2. REFUSAL. Refuse when a question asks about a use, population or dose outside a
   drug's approved indications, and when it asks for individual medical advice about
   a named person rather than general prescribing information. Say plainly that you
   are refusing, say why, cite the policy, and direct the person to their healthcare
   provider.

3. ADVERSE EVENTS. If a question describes a symptom matching a drug's own
   adverse-reaction trigger terms - especially anything suggesting self-harm or a
   severe reaction - escalate before anything else. Set escalate_adverse_event to
   true and direct the person to a healthcare provider or emergency services.

4. PII. Redact names, dates of birth, record numbers, and specific ages combined
   with a relationship. Refer to people generically. Set pii_redacted to true and
   record what you removed in redaction_notes.

Choose disposition to match what you actually did on this turn."""


async def find_prompt_by_name(name: str):
    """The prompt with exactly this name, or None. A notebook helper, NOT an SDK function."""
    found = await hub.prompts.list(search=name, limit=100)
    return next((p for p in found.data if p.name == name), None)


prompt = await find_prompt_by_name(PROMPT)
if prompt is None:
    prompt = await hub.prompts.create(
        name=PROMPT,
        description="Cited, policy-aware medical-information agent with a typed answer shape.",
    )
    print(f"created prompt shell {PROMPT}")
else:
    print(f"prompt {PROMPT} already exists")

if (await hub.prompts.list_versions(prompt.id, limit=1)).total == 0:
    pv = await hub.prompts.commit_version(
        prompt.id,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": "{{ question }}"},
        ],
        model=MODEL,
    )
    print(f"committed prompt v{pv.version_number} on {pv.model}")
else:
    print("prompt already has a version - nothing committed")

created prompt shell medical-information-qa
committed prompt v1 on gpt-4o-mini


---

## Step 7 — Connect all four tools to the prompt

Four bindings, so one render returns the messages and all four tool schemas together.

### In the dashboard

**Prompts → `medical-information-qa` → Tools tab → + Connect a tool from the catalog**, four
times.

| Field | What to enter |
|---|---|
| **Tools** | `get_drug_profile`, `get_inquiry`, `search_prescribing_info`, `check_safety_policy` |
| **Alias** | `production` for each |
| **Column** | **default** — both `production` and `staging` inherit all four |

![The prompt's Tools tab with all four tools connected in the default column](https://docs.acruxcore.com/img/tutorials/build-a-medical-information-qa-agent/07-prompt-attach-tools.png)

### The same thing in code

**Setup.** `set_tool_binding` replaces the binding for a tool rather than adding a second one, so
running this twice leaves four rows and not eight.

In [11]:
for name in TOOL_NAMES:
    binding = await hub.prompts.set_tool_binding(prompt.id, TOOL_IDS[name],
                                                tool_alias="production")
    print(f"bound {binding.tool_name:>24} @ {binding.tool_alias} -> "
          f"v{binding.resolved_version_number}")

bindings = await hub.prompts.list_tool_bindings(prompt.id)
print("\ntools on this prompt:", sorted(b.tool_name for b in bindings.default))

bound         get_drug_profile @ production -> v1
bound              get_inquiry @ production -> v1
bound  search_prescribing_info @ production -> v1
bound      check_safety_policy @ production -> v2

tools on this prompt: ['check_safety_policy', 'get_drug_profile', 'get_inquiry', 'search_prescribing_info']


---

## Step 8 — Run three questions through one schema

**Your app.** This is the part that ships. One call per question, with the tools and the schema
set together, and the answer comes back as JSON you can index into.

Read `ANSWER_SCHEMA` first. Every property is in `required` and `additionalProperties` is
`false`, because `strict: true` demands both — Step 11 shows what happens otherwise.

The three questions are chosen to make the model reach three different dispositions. Nothing in
the code tells it which; that is the whole demonstration.

In [12]:
ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "disposition": {
            "type": "string",
            "enum": ["answer", "answer_with_limitations", "refuse_off_label",
                     "refuse_personal_advice", "escalate_adverse_event"],
        },
        "answer": {"type": "string"},
        "safety_flags": {
            "type": "array",
            "items": {"type": "string",
                      "enum": ["off_label", "personal_medical_advice", "adverse_event",
                               "pii_redacted", "unsupported_claim"]},
        },
        "escalate_adverse_event": {"type": "boolean"},
        "pii_redacted": {"type": "boolean"},
        "redaction_notes": {"type": "array", "items": {"type": "string"}},
        "citations": {"type": "array", "items": {"type": "string"}},
    },
    # strict mode: every property listed, and no extras allowed.
    "required": ["disposition", "answer", "safety_flags", "escalate_adverse_event",
                 "pii_redacted", "redaction_notes", "citations"],
    "additionalProperties": False,
}

RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "medical_information_answer",
        "schema": ANSWER_SCHEMA,
        "strict": True,
    },
}

#: Keyed by catalog tool name. The catalog holds the schemas and no bodies.
CLIENT_TOOLS = {
    "get_drug_profile": get_drug_profile,
    "get_inquiry": get_inquiry,
    "search_prescribing_info": search_prescribing_info,
    "check_safety_policy": check_safety_policy,
}


async def ask(question: str, response_format=RESPONSE_FORMAT) -> dict:
    """Answer one question, shaped. Returns the parsed JSON plus the trace id."""
    rendered = await hub.prompts.render(PROMPT, "production", {"question": question})
    result = await hub.gateway.run_prompt_with_tools(
        rendered,
        client_tools=CLIENT_TOOLS,
        response_format=response_format,
        trace={"name": PROMPT, "session_id": "medical-qa-demo"},
    )
    return {"parsed": json.loads(result.content), "trace_id": result.trace_id,
            "turns": result.iterations}


QUESTIONS = {
    "normal": "What is Cortiblex approved to treat, and is it safe for someone with a "
              "fungal infection?",
    "off_label": "Can I give my 10-year-old Neuravex for anxiety before her school exams?",
    "adverse_event": "I've been having thoughts of hurting myself since I started Neuravex "
                     "two weeks ago. What should I do?",
}

RUNS = {}
for label, question in QUESTIONS.items():
    RUNS[label] = await ask(question)
    answer = RUNS[label]["parsed"]
    print(f"=== {label} ({RUNS[label]['turns']} turns) ===")
    print(f"disposition:  {answer['disposition']}")
    print(f"escalate:     {answer['escalate_adverse_event']}   "
          f"pii_redacted: {answer['pii_redacted']}")
    print(f"safety_flags: {answer['safety_flags']}")
    print(f"citations:    {answer['citations']}")
    print(f"answer:       {answer['answer'][:180]}...")
    print()

=== normal (2 turns) ===
disposition:  refuse_personal_advice
escalate:     False   pii_redacted: False
safety_flags: ['personal_medical_advice', 'off_label']
citations:    ['cortiblex-pi.md']
answer:       Cortiblex is approved for the short-course treatment (up to 14 days) of moderate-to-severe rheumatoid arthritis flare in adults and as adjunct therapy for acute severe allergic rea...

=== off_label (3 turns) ===
disposition:  refuse_off_label
escalate:     False   pii_redacted: False
safety_flags: ['off_label', 'personal_medical_advice']
citations:    ['neuravex-pi.md#section-approved-indications']
answer:       I cannot provide guidance on giving Neuravex to your 10-year-old for anxiety before school exams. Neuravex is only approved for chronic diabetic peripheral neuropathic pain and maj...

=== adverse_event (2 turns) ===
disposition:  escalate_adverse_event
escalate:     True   pii_redacted: False
safety_flags: ['adverse_event']
citations:    []
answer:       I am really sorry 

Three different dispositions, from one schema and one prompt. Your code read
`answer["disposition"]` — it did not search a paragraph for the word "cannot".

Note that `normal`, `off_label` and `adverse_event` are **my names for the questions**, not
predictions. The model picks the disposition, and it does not always pick the one the label
suggests: asking whether a drug "is safe for someone with a fungal infection" can read as a
question about a specific person, and come back `refuse_personal_advice` rather than `answer`.
Whether that is correct behaviour is a judgement call — which is exactly the kind of judgement an
eval exists to make, and a schema cannot.

The wording changes on every run. The dispositions move less, and they are the thing worth
measuring.

**Check.** The citations are checkable, so check them. Every one should name a real `## ` section
in a real fixture file, because `search_prescribing_info` built them from the documents rather
than the model inventing them.

A citation that does not resolve is the model having written its own. The schema cannot prevent
that, because a schema constrains shape and not truth — this is the Step 1 trap, in the one place
you can actually catch it.

What this looks like in practice is a **near miss**, never something obviously wrong. Across runs
of this exact notebook the model produced all three of these:

- underscores where the real slug has hyphens — `#approved_indications`
- an invented prefix — `#section-approved-indications`
- the file name with the anchor dropped entirely — `cortiblex-pi.md`

Every one of them looks right at a glance, satisfies the schema completely, and points nowhere.
The tool returned the correct string; the model retyped it from memory instead of copying it.

If citations matter in your product, this check is not optional, and the fix is to verify the
anchor against the documents rather than to ask the model more firmly.

In [13]:
real_sections = {f"{s['file']}#{s['slug']}" for s in _load_sections(PI_FILES)}
files_only = {s["file"] for s in _load_sections(PI_FILES)}

for label, run in RUNS.items():
    print(f"{label}:")
    for citation in run["parsed"]["citations"]:
        if citation in real_sections:
            verdict = "resolves to a real section"
        elif citation in files_only:
            verdict = "names a real file, no section anchor"
        else:
            verdict = "DOES NOT RESOLVE"
        print(f"  {citation:<48} {verdict}")

normal:
  cortiblex-pi.md                                  names a real file, no section anchor
off_label:
  neuravex-pi.md#section-approved-indications      DOES NOT RESOLVE
adverse_event:


---

## Step 9 — The same schema as a pydantic class

**Your app.** Same behaviour, typed. `Field(description=...)` hints reach the model as per-field
guidance that the hand-written dict has no place for.

One thing to know about what `pydantic_response_format` returns: it is **not** the wire dict. It
is a small marker holding your class, and the SDK calls `model_json_schema()` on it at send time,
importing pydantic only then. So pydantic stays an optional dependency for everyone who does not
use this form.

Use it when you want the type in your own codebase too — the answer can then be validated into a
real object rather than left as a dict.

**Keep the class in its own cell.** A cell containing a top-level `await` is compiled into a
coroutine, so everything defined in it lands in a function scope rather than the notebook's
globals. Pydantic resolves a field's type from module globals when it needs to, so a class defined
in the same cell as an `await` can fail later with *"is not fully defined; you should define
Literal"*. Definitions in one cell, `await` in the next.

In [14]:
from typing import Literal

from pydantic import BaseModel, Field

from acruxcore import pydantic_response_format


class MedicalInformationAnswer(BaseModel):
    """The one shape every answer from this agent takes."""

    disposition: Literal[
        "answer", "answer_with_limitations", "refuse_off_label",
        "refuse_personal_advice", "escalate_adverse_event",
    ] = Field(description="The agent's decision: answer, refuse, or escalate.")
    answer: str = Field(description="The text the end user sees.")
    safety_flags: list[str] = Field(description="Policy flags triggered on this turn.")
    escalate_adverse_event: bool = Field(
        description="True when the question describes a suspected adverse event.")
    pii_redacted: bool = Field(description="True when PII was found and redacted.")
    redaction_notes: list[str] = Field(description="What was redacted and why.")
    citations: list[str] = Field(description="Source references, as filename.md#section-slug.")


PYDANTIC_FORMAT = pydantic_response_format(
    MedicalInformationAnswer, name="medical_information_answer")

print("what the helper returns:", list(PYDANTIC_FORMAT))
print("it holds the class itself:",
      PYDANTIC_FORMAT["__acruxcore_pydantic_response_format__"].__name__)

what the helper returns: ['__acruxcore_pydantic_response_format__', 'name', 'strict']
it holds the class itself: MedicalInformationAnswer


**Your app.** Now run it — a separate cell, because of the `await`.

The answer comes back as JSON exactly as before, and this time it is validated into a real
`MedicalInformationAnswer` object, so a typo in a field name is a type error rather than a
`KeyError` in production.

In [15]:
typed_run = await ask(QUESTIONS["normal"], response_format=PYDANTIC_FORMAT)
typed = MedicalInformationAnswer(**typed_run["parsed"])       # validated into a real object

print(f"disposition: {typed.disposition}")
print(f"citations:   {typed.citations}")
print(f"type:        {type(typed).__name__}, not a dict")

disposition: refuse_personal_advice
citations:   ['cortiblex-pi.md#approved-indications', 'cortiblex-pi.md#contraindication_tags']
type:        MedicalInformationAnswer, not a dict


**Check.** The claim in Step 1 was that both forms produce the same wire dict. Compare them
rather than trusting it.

The marker has to be resolved first, and `normalize_response_format` is the SDK's own function for
that — the same one the gateway call uses internally. Comparing the marker directly would compare
nothing.

The two are not byte-identical, and the difference is informative: pydantic carries each field's
`description` through, and the hand-written dict has none. The *structure* — the properties, the
required list, `additionalProperties`, `strict` — is what has to match.

In [16]:
from acruxcore.response_format import normalize_response_format


def structure_of(response_format: dict) -> dict:
    """The parts of a response_format that decide provider behaviour, ignoring prose."""
    response_format = normalize_response_format(response_format)
    schema = response_format["json_schema"]["schema"]
    return {
        "name": response_format["json_schema"]["name"],
        "strict": response_format["json_schema"].get("strict"),
        "additionalProperties": schema.get("additionalProperties"),
        "properties": sorted(schema["properties"]),
        "required": sorted(schema["required"]),
    }


dict_form = structure_of(RESPONSE_FORMAT)
pydantic_form = structure_of(PYDANTIC_FORMAT)

print("dict form    :", json.dumps(dict_form, indent=2)[:300])
print("\nsame structure:", dict_form == pydantic_form)

wire = normalize_response_format(PYDANTIC_FORMAT)
descriptions = {
    field: spec.get("description")
    for field, spec in wire["json_schema"]["schema"]["properties"].items()
}
print("\nper-field descriptions only pydantic sent:")
for field, description in list(descriptions.items())[:3]:
    print(f"  {field}: {description}")

dict form    : {
  "name": "medical_information_answer",
  "strict": true,
  "additionalProperties": false,
  "properties": [
    "answer",
    "citations",
    "disposition",
    "escalate_adverse_event",
    "pii_redacted",
    "redaction_notes",
    "safety_flags"
  ],
  "required": [
    "answer",
    "citatio

same structure: True

per-field descriptions only pydantic sent:
  disposition: The agent's decision: answer, refuse, or escalate.
  answer: The text the end user sees.
  safety_flags: Policy flags triggered on this turn.


---

## Step 10 — Read the traces back

**Check.** Which tools did the model actually reach for on each question? That is not visible in
the answers, and it is the difference between a grounded answer and a lucky one.

Two mechanics worth knowing, both learned the hard way:

- **Flush first.** Spans are reported in the background so they never slow a request down. Read a
  trace immediately after a run and the tool spans may still be in the queue.
- **Spans are a tree.** A tool span is a *child* of the model turn that asked for it, so counting
  them means walking `.children` rather than filtering the top level.

![A trace for the adverse-event question](https://docs.acruxcore.com/img/tutorials/build-a-medical-information-qa-agent/09-trace-adverse-event.png)

In [17]:
await hub.gateway.flush()          # drain the background span queue before reading


def every_span(spans):
    """Flatten the span tree. A tool span is a child of the model turn that asked for it."""
    for span in spans:
        yield span
        yield from every_span(span.children)


for label, run in RUNS.items():
    detail = await hub.traces.get(run["trace_id"])
    spans = list(every_span(detail.spans))
    tools_used = [s.name for s in spans if s.kind == "tool"]
    print(f"{label:>14}: spans={detail.trace.span_count}  "
          f"tokens={detail.trace.total_tokens}")
    print(f"{'':>14}  tools the model chose: {tools_used}")

        normal: spans=5  tokens=2532
                tools the model chose: ['get_drug_profile', 'check_safety_policy']
     off_label: spans=6  tokens=3129
                tools the model chose: ['check_safety_policy', 'get_drug_profile']
 adverse_event: spans=4  tokens=1982
                tools the model chose: ['check_safety_policy']


Look at which questions pulled in `check_safety_policy`. The model was told to read the policy
before answering anything that might touch it, and whether it obeys is a behaviour you can now
see rather than assume.

Token counts and the exact tool order move on every run. The dispositions and the presence of
grounding tool calls are what should be stable.

---

## Step 11 — Four ways to get this wrong

Every cell in this step is **broken on purpose**. None of it is app code.

### Mistake 1 — no `response_format` at all

**Broken on purpose.** This is the state you are trying to leave behind. Same prompt, same tools,
same question — and an answer your program cannot read without guessing.

In [18]:
rendered = await hub.prompts.render(PROMPT, "production",
                                    {"question": QUESTIONS["off_label"]})
loose = await hub.gateway.run_prompt_with_tools(
    rendered,
    client_tools=CLIENT_TOOLS,
    # Broken on purpose: no response_format, so the model returns prose.
    trace={"name": PROMPT, "session_id": "medical-qa-demo"},
)

print(loose.content[:320], "...\n")
try:
    json.loads(loose.content)
    print("parsed as JSON - you got lucky this run")
except json.JSONDecodeError as err:
    print(f"json.JSONDecodeError: {err}")
    print("There is no disposition field to read. Your only options are to grep the prose")
    print("or to ask another model what it said - both of which are guesses.")

I refuse to answer your question about giving Neuravex to your child, as it asks about a use outside the drug's approved indications. Neuravex is approved only for chronic diabetic peripheral neuropathic pain and major depressive disorder in adults. For any concerns or questions about your child's anxiety and possible  ...

json.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
There is no disposition field to read. Your only options are to grep the prose
or to ask another model what it said - both of which are guesses.


### Mistake 2 — a property missing from `required`

**Broken on purpose.** Under `strict: true` there is no such thing as an optional field. Leaving
one out of `required` is rejected before the model is even called, which is the good kind of
failure.

Read the error carefully, because it will not tell you which rule you broke. It says the provider
rejected the request, not *why*. So learn the two rules — every property in `required`, and
`additionalProperties: false` — because a 400 here means one of them, and you have to work out
which.

In [19]:
from acruxcore.errors import AcruxCoreError

incomplete = json.loads(json.dumps(ANSWER_SCHEMA))       # a deep copy
incomplete["required"] = ["disposition", "answer"]       # broken on purpose: the rest omitted

try:
    await hub.gateway.chat(
        MODEL,
        [{"role": "user", "content": "What is Cortiblex for?"}],
        response_format={"type": "json_schema", "json_schema": {
            "name": "medical_information_answer", "schema": incomplete, "strict": True}},
        trace=False,
    )
    print("no error - unexpected")
except AcruxCoreError as err:
    print(f"{err.status_code} {err.code}")
    # str(err) is a summary; .body carries the gateway's own error object.
    print("  ", json.dumps(err.body))

400 API_ERROR
   {"error": {"code": "PROVIDER_BAD_REQUEST", "message": "Provider rejected the request (400): OpenAI request failed with status 400"}}


### Mistake 3 — `additionalProperties` left out under strict mode

**Broken on purpose.** The other half of the same rule. Strict mode has to know that no extra keys
are allowed, and it will not assume it.

In [20]:
loose_schema = json.loads(json.dumps(ANSWER_SCHEMA))
del loose_schema["additionalProperties"]                 # broken on purpose

try:
    await hub.gateway.chat(
        MODEL,
        [{"role": "user", "content": "What is Cortiblex for?"}],
        response_format={"type": "json_schema", "json_schema": {
            "name": "medical_information_answer", "schema": loose_schema, "strict": True}},
        trace=False,
    )
    print("no error - unexpected")
except AcruxCoreError as err:
    print(f"{err.status_code} {err.code}")
    # str(err) is a summary; .body carries the gateway's own error object.
    print("  ", json.dumps(err.body))

400 API_ERROR
   {"error": {"code": "PROVIDER_BAD_REQUEST", "message": "Provider rejected the request (400): OpenAI request failed with status 400"}}


### Mistake 4 — adding a docstring to the tool the dashboard owns

**Broken on purpose.** This is the flip side of Step 5, and it is the quiet one. Give
`check_safety_policy` a docstring and the next sync now *does* send a description — which
replaces the compliance reviewer's wording. No error, no warning, and nobody notices until the
model starts calling the tool differently.

The cell restores the approved wording afterwards, so the notebook does not leave your catalog in
that state.

In [21]:
tools_api._sync_cache.clear()

before = (await hub.tools.resolve(
    [{"name": "check_safety_policy", "alias": "production"}]))[0].function.get("description")

# Broken on purpose: a docstring where there deliberately was none. The decorator reads
# the docstring at decoration time, so re-decorating rebuilds the spec from it.
check_safety_policy.__doc__ = "Checks the safety policy."
check_safety_policy = acrux.tool(check_safety_policy)
clobbered = await hub.tools.sync([check_safety_policy])
after = (await hub.tools.resolve(
    [{"name": "check_safety_policy", "alias": "production"}]))[0].function.get("description")

print(f"v{clobbered[0].version_number}, committed={clobbered[0].committed}")
print(f"before: {(before or '')[:70]}...")
print(f"after:  {(after or '')[:70]}...")
print(f"the compliance wording survived: {after == COMPLIANCE_DESCRIPTION}")

# Put it back: remove the docstring again, which is the Step 5 rule.
check_safety_policy.__doc__ = None
check_safety_policy = acrux.tool(check_safety_policy)
tools_api._sync_cache.clear()
restored = await hub.tools.commit_version(
    policy_tool_id,
    parameters_schema=(await hub.tools.resolve(
        [{"name": "check_safety_policy", "alias": "production"}]))[0].function["parameters"],
    executor={"type": "client"},
    description=COMPLIANCE_DESCRIPTION,
    changelog="Restore compliance-approved wording.",
)
await hub.tools.promote_alias(policy_tool_id, "production", restored.version_number)
print(f"\nrestored on v{restored.version_number}")

v3, committed=True
before: Retrieve the team's approved safety-policy text for one topic: respons...
after:  Checks the safety policy....
the compliance wording survived: False

restored on v4


---

## Step 12 — Close the clients

**Your app.** Traces are reported in the background so they never slow your request down. Closing
the client flushes whatever is still queued. In a script `async with AcruxCore() as hub:` does it
for you; a notebook has no block to leave, so do it by hand.

The raw `rest` client from the preflight needs closing too.

In [22]:
await hub.gateway.aclose()
await rest.aclose()
print("flushed")

flushed


---

## What you built

An agent whose answers are data. It decided on its own whether each question was answerable,
off-label or an emergency, cited the documents it used, and returned all of that in a shape your
program can branch on without reading a word of it.

### What of this actually ships

The four implementations from Step 3, plus this:

```python
import json, os
from acruxcore import AcruxCore

ANSWER_SCHEMA = {...}          # exactly as in Step 8
RESPONSE_FORMAT = {"type": "json_schema", "json_schema": {
    "name": "medical_information_answer", "schema": ANSWER_SCHEMA, "strict": True}}

CLIENT_TOOLS = {
    "get_drug_profile": get_drug_profile,
    "get_inquiry": get_inquiry,
    "search_prescribing_info": search_prescribing_info,
    "check_safety_policy": check_safety_policy,
}


async def ask(question: str) -> dict:
    async with AcruxCore() as hub:
        rendered = await hub.prompts.render("medical-information-qa", "production",
                                            {"question": question})
        result = await hub.gateway.run_prompt_with_tools(
            rendered,
            client_tools=CLIENT_TOOLS,
            response_format=RESPONSE_FORMAT,
            trace={"name": "medical-information-qa", "session_id": "medical-qa-demo"},
        )
        return json.loads(result.content)


answer = await ask("What is Cortiblex approved to treat?")
if answer["escalate_adverse_event"]:
    page_the_safety_team(answer)          # your own code, reading one boolean
```

That last `if` is the whole reason for the schema.

Everything else was scaffolding:

- `find_prompt_by_name` exists so this notebook can be re-run. It is a notebook helper, not an
  SDK call.
- the fixture-writing cell stands in for documents you already have.
- the create-and-commit cells are the dashboard's job, done once.
- every **Check** cell — the preflight, the citation check, the structure comparison, the trace
  read — proves a step worked. None of it belongs in a request path.
- Step 11 is all deliberately broken.

### What the schema still cannot do

`strict: true` gave you valid JSON with every field present and only the enum values you allowed.
It gave you no guarantee that `disposition` is *correct*, or that a citation points somewhere
real. That is why Step 8 ends with a citation check and why `disposition` deserves an eval — see
[Evaluate a prompt against a dataset](https://docs.acruxcore.com/docs/guides/evaluate-a-prompt-against-a-dataset).

### What this notebook left in your team

- four tools, all `client` executors; `check_safety_policy` on a dashboard-authored description
- a prompt `medical-information-qa` at v1, with a `question` variable and the bound model
- four bindings, inherited by every prompt alias
- a session `medical-qa-demo` holding every run above, including the unshaped one

### Where to go next

- [Evaluate a prompt against a dataset](https://docs.acruxcore.com/docs/guides/evaluate-a-prompt-against-a-dataset)
  — measuring how often `disposition` is right, which the schema cannot do for you.
- [Create a tool](https://docs.acruxcore.com/docs/guides/create-a-tool)
  — the ownership split from Step 5, compared properly.
- [Using sessions and traces](https://docs.acruxcore.com/docs/guides/using-sessions-and-traces)
  — group related runs and dig into what happened.